# 04 Read Results (Part 2 / RegGS)

Goal:

1. scan the current `output/part2/` tree directly
2. detect complete Part 2 runs with three files: `eval_train.json`, `eval_test.json`, `ate_aligned.json`
3. collect run-level metrics (train/test averages + ATE aligned stats)
4. collect frame-level metrics from train/test json files
5. export clean csv summaries into `results/part2/`

Notes:
- Part 2 has no convergence logs in this protocol.
- ATE values are kept, but cross-dataset ATE comparison is marked as not recommended.

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

CV_ROOT = Path('~/CV_Project').expanduser()
OUTPUT_ROOT = CV_ROOT / 'output' / 'part2'
RESULTS_ROOT = CV_ROOT / 'results' / 'part2'

FINAL_DIR = RESULTS_ROOT / 'final'
FRAME_DIR = RESULTS_ROOT / 'frame'
QC_DIR = RESULTS_ROOT / 'qc'

for d in [FINAL_DIR, FRAME_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('RESULTS_ROOT =', RESULTS_ROOT)
print('FINAL_DIR =', FINAL_DIR)
print('FRAME_DIR =', FRAME_DIR)
print('QC_DIR =', QC_DIR)

OUTPUT_ROOT = /home/bzhang512/CV_Project/output/part2
RESULTS_ROOT = /home/bzhang512/CV_Project/results/part2
FINAL_DIR = /home/bzhang512/CV_Project/results/part2/final
FRAME_DIR = /home/bzhang512/CV_Project/results/part2/frame
QC_DIR = /home/bzhang512/CV_Project/results/part2/qc


## 1. Scan run directories and build run inventory

In [2]:
required_files = ['eval_train.json', 'eval_test.json', 'ate_aligned.json']

dataset_dirs = sorted([d for d in OUTPUT_ROOT.iterdir() if d.is_dir()]) if OUTPUT_ROOT.exists() else []
candidate_run_dirs = []
for dataset_dir in dataset_dirs:
    run_dirs = sorted([d for d in dataset_dir.iterdir() if d.is_dir()])
    candidate_run_dirs.extend(run_dirs)

inventory_rows = []
for run_dir in candidate_run_dirs:
    dataset = run_dir.parent.name
    run_name = run_dir.name
    rel_run_dir = run_dir.relative_to(OUTPUT_ROOT)

    exists_flags = {f"exists_{name.replace('.json', '')}": (run_dir / name).exists() for name in required_files}
    missing_files = [name for name in required_files if not (run_dir / name).exists()]
    is_complete = len(missing_files) == 0

    sr_match = re.search(r'sr(\d+)', run_name)
    nv_match = re.search(r'nv(\d+)', run_name)

    inventory_rows.append({
        'dataset': dataset,
        'run_name': run_name,
        'experiment_name': f'{dataset}__{run_name}',
        'run_dir': str(run_dir),
        'run_dir_rel': str(rel_run_dir),
        'sample_rate_from_name': int(sr_match.group(1)) if sr_match else np.nan,
        'n_views_from_name': int(nv_match.group(1)) if nv_match else np.nan,
        **exists_flags,
        'missing_files': ';'.join(missing_files),
        'is_complete': is_complete,
        'skip_reason': '' if is_complete else f"missing:{';'.join(missing_files)}",
    })

inventory_df = pd.DataFrame(inventory_rows)
if inventory_df.empty:
    inventory_df = pd.DataFrame(columns=[
        'dataset', 'run_name', 'experiment_name', 'run_dir', 'run_dir_rel',
        'sample_rate_from_name', 'n_views_from_name',
        'exists_eval_train', 'exists_eval_test', 'exists_ate_aligned',
        'missing_files', 'is_complete', 'skip_reason'
    ])

complete_runs_df = inventory_df[inventory_df['is_complete']].copy()
skipped_runs_df = inventory_df[~inventory_df['is_complete']].copy()

print('num candidate runs =', len(inventory_df))
print('num complete runs =', len(complete_runs_df))
print('num skipped runs =', len(skipped_runs_df))
inventory_df.sort_values(['dataset', 'run_name']).head(20)

num candidate runs = 5
num complete runs = 3
num skipped runs = 2


,dataset,run_name,experiment_name,run_dir,run_dir_rel,sample_rate_from_name,n_views_from_name,exists_eval_train,exists_eval_test,exists_ate_aligned,missing_files,is_complete,skip_reason
0,405841,reggs_405841_scene_A_sr10_nv4,405841__reggs_405841_scene_A_sr10_nv4,/home/bzhang512/CV_Project/output/part2/405841...,405841/reggs_405841_scene_A_sr10_nv4,10.0,4.0,True,True,True,,True,
1,dl3dv_2,reggs_dl3dv2_sr30_nv8,dl3dv_2__reggs_dl3dv2_sr30_nv8,/home/bzhang512/CV_Project/output/part2/dl3dv_...,dl3dv_2/reggs_dl3dv2_sr30_nv8,30.0,8.0,True,True,True,,True,
2,re10k,reggs_re10k1_sr30_nv8,re10k__reggs_re10k1_sr30_nv8,/home/bzhang512/CV_Project/output/part2/re10k/...,re10k/reggs_re10k1_sr30_nv8,30.0,8.0,True,True,True,,True,
3,reggs_smoke,reggs_dl3dv2_fullseq_256,reggs_smoke__reggs_dl3dv2_fullseq_256,/home/bzhang512/CV_Project/output/part2/reggs_...,reggs_smoke/reggs_dl3dv2_fullseq_256,NaN,NaN,False,False,False,eval_train.json;eval_test.json;ate_aligned.json,False,missing:eval_train.json;eval_test.json;ate_ali...
4,reggs_smoke,reggs_re10k1_fullseq_256,reggs_smoke__reggs_re10k1_fullseq_256,/home/bzhang512/CV_Project/output/part2/reggs_...,reggs_smoke/reggs_re10k1_fullseq_256,NaN,NaN,False,False,False,eval_train.json;eval_test.json;ate_aligned.json,False,missing:eval_train.json;eval_test.json;ate_ali...


## 2. Parse run-level and frame-level metrics from the three json files

In [3]:
def read_json_safe(path: Path):
    if not path.exists():
        return None, f'missing:{path.name}'
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
        if not isinstance(payload, dict):
            return None, f'invalid_root:{path.name}'
        return payload, 'ok'
    except Exception as exc:
        return None, f'json_error:{path.name}:{exc.__class__.__name__}'

def as_float(value):
    try:
        if value is None:
            return np.nan
        return float(value)
    except Exception:
        return np.nan

def ensure_metric_list(payload):
    if not isinstance(payload, dict):
        return []
    metrics = payload.get('metrics', [])
    return metrics if isinstance(metrics, list) else []

run_rows = []
frame_rows = []

for _, inv in complete_runs_df.iterrows():
    run_dir = Path(inv['run_dir'])
    dataset = inv['dataset']
    run_name = inv['run_name']
    experiment_name = inv['experiment_name']

    train_payload, train_status = read_json_safe(run_dir / 'eval_train.json')
    test_payload, test_status = read_json_safe(run_dir / 'eval_test.json')
    ate_payload, ate_status = read_json_safe(run_dir / 'ate_aligned.json')

    train_metrics = ensure_metric_list(train_payload)
    test_metrics = ensure_metric_list(test_payload)

    run_rows.append({
        'dataset': dataset,
        'run_name': run_name,
        'experiment_name': experiment_name,
        'run_dir': str(run_dir),
        'sample_rate_from_name': inv['sample_rate_from_name'],
        'n_views_from_name': inv['n_views_from_name'],
        'train_json_status': train_status,
        'test_json_status': test_status,
        'ate_json_status': ate_status,
        'train_avg_lpips': as_float((train_payload or {}).get('avg_lpips')),
        'train_avg_psnr': as_float((train_payload or {}).get('avg_psnr')),
        'train_avg_ssim': as_float((train_payload or {}).get('avg_ssim')),
        'test_avg_lpips': as_float((test_payload or {}).get('avg_lpips')),
        'test_avg_psnr': as_float((test_payload or {}).get('avg_psnr')),
        'test_avg_ssim': as_float((test_payload or {}).get('avg_ssim')),
        'n_train_frames': len(train_metrics),
        'n_test_frames': len(test_metrics),
        'ate_compared_pose_pairs': as_float((ate_payload or {}).get('compared_pose_pairs')),
        'ate_rmse': as_float((ate_payload or {}).get('rmse')),
        'ate_mean': as_float((ate_payload or {}).get('mean')),
        'ate_median': as_float((ate_payload or {}).get('median')),
        'ate_std': as_float((ate_payload or {}).get('std')),
        'ate_min': as_float((ate_payload or {}).get('min')),
        'ate_max': as_float((ate_payload or {}).get('max')),
    })

    for split, metrics in [('train', train_metrics), ('test', test_metrics)]:
        for metric in metrics:
            frame_id = metric.get('frame_id', None) if isinstance(metric, dict) else None
            frame_id_str = '' if frame_id is None else str(frame_id)
            frame_rows.append({
                'dataset': dataset,
                'run_name': run_name,
                'experiment_name': experiment_name,
                'split': split,
                'frame_id': frame_id_str,
                'frame_index': pd.to_numeric(frame_id_str, errors='coerce'),
                'lpips': as_float(metric.get('lpips') if isinstance(metric, dict) else None),
                'psnr': as_float(metric.get('psnr') if isinstance(metric, dict) else None),
                'ssim': as_float(metric.get('ssim') if isinstance(metric, dict) else None),
                'run_dir': str(run_dir),
            })

run_summary_df = pd.DataFrame(run_rows)
frame_df = pd.DataFrame(frame_rows)

if run_summary_df.empty:
    run_summary_df = pd.DataFrame(columns=[
        'dataset', 'run_name', 'experiment_name', 'run_dir',
        'sample_rate_from_name', 'n_views_from_name',
        'train_json_status', 'test_json_status', 'ate_json_status',
        'train_avg_lpips', 'train_avg_psnr', 'train_avg_ssim',
        'test_avg_lpips', 'test_avg_psnr', 'test_avg_ssim',
        'n_train_frames', 'n_test_frames',
        'ate_compared_pose_pairs', 'ate_rmse', 'ate_mean', 'ate_median',
        'ate_std', 'ate_min', 'ate_max'
    ])

if frame_df.empty:
    frame_df = pd.DataFrame(columns=[
        'dataset', 'run_name', 'experiment_name', 'split',
        'frame_id', 'frame_index', 'lpips', 'psnr', 'ssim', 'run_dir'
    ])

print('run_summary_df shape =', run_summary_df.shape)
print('frame_df shape =', frame_df.shape)
run_summary_df.head()

run_summary_df shape = (3, 24)
frame_df shape = (41, 10)


,dataset,run_name,experiment_name,run_dir,sample_rate_from_name,n_views_from_name,train_json_status,test_json_status,ate_json_status,train_avg_lpips,...,test_avg_ssim,n_train_frames,n_test_frames,ate_compared_pose_pairs,ate_rmse,ate_mean,ate_median,ate_std,ate_min,ate_max
0,405841,reggs_405841_scene_A_sr10_nv4,405841__reggs_405841_scene_A_sr10_nv4,/home/bzhang512/CV_Project/output/part2/405841...,10.0,4.0,ok,ok,ok,0.079666,...,0.561145,4,2,4.0,0.661653,0.612167,0.541786,0.251070,0.346177,1.018922
1,dl3dv_2,reggs_dl3dv2_sr30_nv8,dl3dv_2__reggs_dl3dv2_sr30_nv8,/home/bzhang512/CV_Project/output/part2/dl3dv_...,30.0,8.0,ok,ok,ok,0.001365,...,0.446930,8,10,8.0,2.325984,1.937354,1.626277,1.287191,0.551933,4.996962
2,re10k,reggs_re10k1_sr30_nv8,re10k__reggs_re10k1_sr30_nv8,/home/bzhang512/CV_Project/output/part2/re10k/...,30.0,8.0,ok,ok,ok,0.004270,...,0.879175,8,9,8.0,0.010643,0.009655,0.009389,0.004477,0.001974,0.017910


## 3. Add generalization-gap fields and ATE comparison notes

In [4]:
if run_summary_df.empty:
    dataset_summary_df = pd.DataFrame(columns=[
        'dataset', 'num_runs',
        'mean_train_avg_psnr', 'mean_test_avg_psnr', 'mean_psnr_drop_train_to_test',
        'mean_train_avg_ssim', 'mean_test_avg_ssim', 'mean_ssim_drop_train_to_test',
        'mean_train_avg_lpips', 'mean_test_avg_lpips', 'mean_lpips_increase_train_to_test',
        'mean_ate_rmse', 'mean_ate_mean',
        'total_train_frames', 'total_test_frames',
        'ate_cross_dataset_comparison'
    ])
else:
    run_summary_df['psnr_drop_train_to_test'] = run_summary_df['train_avg_psnr'] - run_summary_df['test_avg_psnr']
    run_summary_df['ssim_drop_train_to_test'] = run_summary_df['train_avg_ssim'] - run_summary_df['test_avg_ssim']
    run_summary_df['lpips_increase_train_to_test'] = run_summary_df['test_avg_lpips'] - run_summary_df['train_avg_lpips']

    high_risk = (
        (run_summary_df['psnr_drop_train_to_test'] > 10.0)
        | (run_summary_df['ssim_drop_train_to_test'] > 0.10)
        | (run_summary_df['lpips_increase_train_to_test'] > 0.10)
    )
    run_summary_df['generalization_risk'] = np.where(high_risk, 'high_gap', 'low_or_unknown')

    run_summary_df['ate_compare_scope'] = 'within_dataset_only'
    run_summary_df['ate_compare_note'] = (
        'ATE scale can differ by dataset. Compare ATE RMSE only within the same dataset.'
    )

    dataset_summary_df = (
        run_summary_df.groupby('dataset', dropna=False)
        .agg(
            num_runs=('experiment_name', 'nunique'),
            mean_train_avg_psnr=('train_avg_psnr', 'mean'),
            mean_test_avg_psnr=('test_avg_psnr', 'mean'),
            mean_psnr_drop_train_to_test=('psnr_drop_train_to_test', 'mean'),
            mean_train_avg_ssim=('train_avg_ssim', 'mean'),
            mean_test_avg_ssim=('test_avg_ssim', 'mean'),
            mean_ssim_drop_train_to_test=('ssim_drop_train_to_test', 'mean'),
            mean_train_avg_lpips=('train_avg_lpips', 'mean'),
            mean_test_avg_lpips=('test_avg_lpips', 'mean'),
            mean_lpips_increase_train_to_test=('lpips_increase_train_to_test', 'mean'),
            mean_ate_rmse=('ate_rmse', 'mean'),
            mean_ate_mean=('ate_mean', 'mean'),
            total_train_frames=('n_train_frames', 'sum'),
            total_test_frames=('n_test_frames', 'sum'),
        )
        .reset_index()
    )
    dataset_summary_df['ate_cross_dataset_comparison'] = 'not_recommended'

ate_summary_cols = [
    'dataset', 'run_name', 'experiment_name',
    'ate_compared_pose_pairs', 'ate_rmse', 'ate_mean', 'ate_median',
    'ate_std', 'ate_min', 'ate_max', 'ate_compare_scope', 'ate_compare_note'
]
for col in ate_summary_cols:
    if col not in run_summary_df.columns:
        run_summary_df[col] = np.nan
ate_summary_df = run_summary_df[ate_summary_cols].copy()

print('dataset_summary_df shape =', dataset_summary_df.shape)
run_summary_df[['dataset', 'run_name', 'train_avg_psnr', 'test_avg_psnr', 'psnr_drop_train_to_test', 'generalization_risk']].head()

dataset_summary_df shape = (3, 16)


,dataset,run_name,train_avg_psnr,test_avg_psnr,psnr_drop_train_to_test,generalization_risk
0,405841,reggs_405841_scene_A_sr10_nv4,49.940655,19.048077,30.892578,high_gap
1,dl3dv_2,reggs_dl3dv2_sr30_nv8,52.825459,15.689012,37.136448,high_gap
2,re10k,reggs_re10k1_sr30_nv8,42.275033,25.001969,17.273063,high_gap


## 4. Export csv summaries into results/part2

In [5]:
run_summary_csv = FINAL_DIR / 'part2_run_summary.csv'
ate_summary_csv = FINAL_DIR / 'part2_ate_summary.csv'
dataset_summary_csv = FINAL_DIR / 'part2_dataset_summary.csv'
frame_metrics_csv = FRAME_DIR / 'part2_frame_metrics.csv'
run_inventory_csv = QC_DIR / 'part2_run_inventory.csv'
skipped_runs_csv = QC_DIR / 'part2_skipped_runs.csv'

run_summary_df.sort_values(['dataset', 'run_name']).to_csv(run_summary_csv, index=False)
ate_summary_df.sort_values(['dataset', 'run_name']).to_csv(ate_summary_csv, index=False)
dataset_summary_df.sort_values(['dataset']).to_csv(dataset_summary_csv, index=False)
frame_df.sort_values(['dataset', 'run_name', 'split', 'frame_index', 'frame_id']).to_csv(frame_metrics_csv, index=False)
inventory_df.sort_values(['dataset', 'run_name']).to_csv(run_inventory_csv, index=False)
skipped_runs_df.sort_values(['dataset', 'run_name']).to_csv(skipped_runs_csv, index=False)

print('run_summary_csv =', run_summary_csv)
print('ate_summary_csv =', ate_summary_csv)
print('dataset_summary_csv =', dataset_summary_csv)
print('frame_metrics_csv =', frame_metrics_csv)
print('run_inventory_csv =', run_inventory_csv)
print('skipped_runs_csv =', skipped_runs_csv)

run_summary_csv = /home/bzhang512/CV_Project/results/part2/final/part2_run_summary.csv
ate_summary_csv = /home/bzhang512/CV_Project/results/part2/final/part2_ate_summary.csv
dataset_summary_csv = /home/bzhang512/CV_Project/results/part2/final/part2_dataset_summary.csv
frame_metrics_csv = /home/bzhang512/CV_Project/results/part2/frame/part2_frame_metrics.csv
run_inventory_csv = /home/bzhang512/CV_Project/results/part2/qc/part2_run_inventory.csv
skipped_runs_csv = /home/bzhang512/CV_Project/results/part2/qc/part2_skipped_runs.csv


## 5. Quick checks (coverage, skipped runs, frame counts)

In [7]:
def show_df(df, n=10):
    if df.empty:
        print('empty dataframe')
        return
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string(index=False))

print('candidate runs =', len(inventory_df))
print('complete runs =', len(complete_runs_df))
print('skipped runs =', len(skipped_runs_df))
print('run_summary rows =', len(run_summary_df))
print('frame rows =', len(frame_df))

print('\nRun summary (selected columns):')
cols = [
    'dataset', 'run_name',
    'train_avg_psnr', 'test_avg_psnr', 'psnr_drop_train_to_test',
    'train_avg_ssim', 'test_avg_ssim', 'ssim_drop_train_to_test',
    'train_avg_lpips', 'test_avg_lpips', 'lpips_increase_train_to_test',
    'n_train_frames', 'n_test_frames',
    'ate_rmse', 'generalization_risk'
]
cols = [c for c in cols if c in run_summary_df.columns]
show_df(run_summary_df[cols].sort_values(['dataset', 'run_name']), n=30)

print('\nSkipped runs and reasons:')
show_df(skipped_runs_df[['dataset', 'run_name', 'missing_files', 'skip_reason']].sort_values(['dataset', 'run_name']), n=30)

if not frame_df.empty:
    frame_count_df = (
        frame_df.groupby(['dataset', 'run_name', 'split'], dropna=False)
        .size()
        .reset_index(name='num_frames')
        .sort_values(['dataset', 'run_name', 'split'])
    )
    print('\nFrame counts per run/split:')
    show_df(frame_count_df, n=50)

print('\nATE note: compare ATE RMSE only within the same dataset; avoid direct cross-dataset ranking.')

candidate runs = 5
complete runs = 3
skipped runs = 2
run_summary rows = 3
frame rows = 41

Run summary (selected columns):


,dataset,run_name,train_avg_psnr,test_avg_psnr,psnr_drop_train_to_test,train_avg_ssim,test_avg_ssim,ssim_drop_train_to_test,train_avg_lpips,test_avg_lpips,lpips_increase_train_to_test,n_train_frames,n_test_frames,ate_rmse,generalization_risk
0,405841,reggs_405841_scene_A_sr10_nv4,49.940655,19.048077,30.892578,0.987666,0.561145,0.426521,0.079666,0.587988,0.508322,4,2,0.661653,high_gap
1,dl3dv_2,reggs_dl3dv2_sr30_nv8,52.825459,15.689012,37.136448,0.999331,0.446930,0.552401,0.001365,0.466456,0.465091,8,10,2.325984,high_gap
2,re10k,reggs_re10k1_sr30_nv8,42.275033,25.001969,17.273063,0.997304,0.879175,0.118129,0.004270,0.143630,0.139360,8,9,0.010643,high_gap



Skipped runs and reasons:


,dataset,run_name,missing_files,skip_reason
3,reggs_smoke,reggs_dl3dv2_fullseq_256,eval_train.json;eval_test.json;ate_aligned.json,missing:eval_train.json;eval_test.json;ate_ali...
4,reggs_smoke,reggs_re10k1_fullseq_256,eval_train.json;eval_test.json;ate_aligned.json,missing:eval_train.json;eval_test.json;ate_ali...



Frame counts per run/split:


,dataset,run_name,split,num_frames
0,405841,reggs_405841_scene_A_sr10_nv4,test,2
1,405841,reggs_405841_scene_A_sr10_nv4,train,4
2,dl3dv_2,reggs_dl3dv2_sr30_nv8,test,10
3,dl3dv_2,reggs_dl3dv2_sr30_nv8,train,8
4,re10k,reggs_re10k1_sr30_nv8,test,9
5,re10k,reggs_re10k1_sr30_nv8,train,8



ATE note: compare ATE RMSE only within the same dataset; avoid direct cross-dataset ranking.
